# Multi Model Provider Setup

In this notebook we set up different LLM providers for AI agents.

Providers in this notebook:

- Gemini using `langchain-google-genai`
- Google Vertex AI using `langchain-google-vertexai`
- Mistral AI using `langchain-mistralai`
- Anthropic Claude using `langchain-anthropic`
- xAI Grok using `langchain-xai`

Simple idea: the agent is your app, and the LLM provider is the brain you choose.

## Easy Explanation

Think about a restaurant.

The restaurant can have different chefs. One chef may be fast, one may be better at careful reasoning, one may be cheaper, and one may be better for business use.

In AI agents, different model providers are like different chefs.

Your agent can use:

| Provider | Easy meaning |
| --- | --- |
| Gemini | Google AI model, good for general tasks |
| Vertex AI | Google Cloud version for production/business apps |
| Mistral | Fast European AI model provider |
| Anthropic | Claude models, good for careful writing/reasoning |
| xAI | Grok models from xAI |

We write one helper function so we can switch providers more easily.

## Step 1: Required Packages

These packages were added to `requirements.txt`:

```text
langchain-google-genai
langchain-google-vertexai
langchain-mistralai
langchain-anthropic
langchain-xai
```

After installing them, we can import provider-specific chat models.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"] = "false"

print("Environment loaded")

## Step 2: API Keys Needed

Each provider needs its own API key.

Do not write real keys inside the notebook.

Put keys in your `.env` file like this:

```text
GOOGLE_API_KEY=your_gemini_key_here
MISTRAL_API_KEY=your_mistral_key_here
ANTHROPIC_API_KEY=your_anthropic_key_here
XAI_API_KEY=your_xai_key_here
GOOGLE_CLOUD_PROJECT=your_google_cloud_project
GOOGLE_CLOUD_LOCATION=us-central1
```

For Vertex AI, you may also need Google Cloud authentication on your computer.

In [ ]:
def key_status(env_name: str) -> str:
    return "set" if os.getenv(env_name) else "missing"

keys = {
    "Gemini / Google AI Studio": "GOOGLE_API_KEY",
    "Google Vertex AI project": "GOOGLE_CLOUD_PROJECT",
    "Google Vertex AI location": "GOOGLE_CLOUD_LOCATION",
    "Mistral AI": "MISTRAL_API_KEY",
    "Anthropic Claude": "ANTHROPIC_API_KEY",
    "xAI Grok": "XAI_API_KEY",
}

for label, env_name in keys.items():
    print(f"{label}: {key_status(env_name)}")

## Step 3: Import All Provider Classes

If this cell runs without error, the packages are installed correctly.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_vertexai import ChatVertexAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic
from langchain_xai import ChatXAI

print("All provider imports worked")

## Step 4: Build One Helper Function

This function creates an LLM based on the provider name.

If the required key is missing, it returns `None` instead of crashing.

Model names change over time, so you can replace these model names with the model available in your provider account.

In [ ]:
def build_llm(provider: str):
    provider = provider.lower().strip()

    if provider == "gemini":
        if not os.getenv("GOOGLE_API_KEY"):
            print("GOOGLE_API_KEY is missing")
            return None
        return ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
        )

    if provider == "vertex":
        project = os.getenv("GOOGLE_CLOUD_PROJECT")
        location = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
        if not project:
            print("GOOGLE_CLOUD_PROJECT is missing")
            return None
        return ChatVertexAI(
            model="gemini-1.5-flash",
            project=project,
            location=location,
            temperature=0,
        )

    if provider == "mistral":
        if not os.getenv("MISTRAL_API_KEY"):
            print("MISTRAL_API_KEY is missing")
            return None
        return ChatMistralAI(
            model="mistral-small-latest",
            temperature=0,
            api_key=os.getenv("MISTRAL_API_KEY"),
        )

    if provider == "anthropic":
        if not os.getenv("ANTHROPIC_API_KEY"):
            print("ANTHROPIC_API_KEY is missing")
            return None
        return ChatAnthropic(
            model="claude-3-5-haiku-latest",
            temperature=0,
            api_key=os.getenv("ANTHROPIC_API_KEY"),
        )

    if provider == "xai":
        if not os.getenv("XAI_API_KEY"):
            print("XAI_API_KEY is missing")
            return None
        return ChatXAI(
            model="grok-2-latest",
            temperature=0,
            api_key=os.getenv("XAI_API_KEY"),
        )

    print("Unknown provider. Use: gemini, vertex, mistral, anthropic, or xai")
    return None

## Step 5: Test One Provider

Change `provider_name` to whichever provider key you have.

Examples:

```python
provider_name = "gemini"
provider_name = "mistral"
provider_name = "anthropic"
provider_name = "xai"
provider_name = "vertex"
```

In [ ]:
provider_name = "gemini"

llm = build_llm(provider_name)

if llm:
    response = llm.invoke([HumanMessage(content="Say hello in one short sentence.")])
    print(response.content)
else:
    print("Provider is not ready yet. Add the API key in .env first.")

## Step 6: Compare Providers With The Same Prompt

This cell tries multiple providers.

It skips any provider where the API key is missing.

In [ ]:
def ask_provider(provider: str, question: str):
    llm = build_llm(provider)
    if not llm:
        return f"{provider}: skipped because key/setup is missing"

    try:
        response = llm.invoke([HumanMessage(content=question)])
        return f"{provider}: {response.content}"
    except Exception as error:
        return f"{provider}: error - {error}"

question = "Explain AI agents in one easy sentence."
providers = ["gemini", "mistral", "anthropic", "xai", "vertex"]

for provider in providers:
    print(ask_provider(provider, question))
    print("-" * 80)

## Step 7: Use Provider In An Agent Later

Once we choose the provider, we can use it inside LangGraph or tools.

Example:

```python
llm = build_llm("gemini")
```

Then use `llm` inside your agent node.

In [ ]:
def simple_answer(question: str, provider: str = "gemini") -> str:
    llm = build_llm(provider)
    if not llm:
        return "This provider is not ready. Please add its API key first."

    response = llm.invoke([HumanMessage(content=question)])
    return response.content

# Run after your selected provider key is ready.
# simple_answer("What can an AI agent do for a restaurant?", provider="gemini")

## Which Provider Should I Use?

For learning:

- Start with Groq or Gemini if you have a free/low-cost key.
- Use Mistral if you want another fast model option.
- Use Anthropic if you want careful writing and reasoning.
- Use Vertex AI when a client wants Google Cloud production setup.
- Use xAI when a client specifically wants Grok/xAI.

For your current learning path, you do not need all keys immediately.

The important thing is: your agent code can be designed so the model provider can be switched later.